In [ ]:
!pip install torch torchvision accelerate pillow scikit-learn numpy==1.23.5 transformers datasets captum

# New section

In [ ]:
import torch
import os
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.metrics import precision_recall_fscore_support, confusion_matrix, classification_report
from datasets import load_dataset
from transformers import (
    AutoImageProcessor,
    ViTForImageClassification,
    TrainingArguments,
    Trainer
)
import pandas as pd
import seaborn as sns
from torch.utils.data import DataLoader
from PIL import Image
import torchvision.transforms as transforms
from captum.attr import visualization as viz
from captum.attr import LayerAttribution, LayerGradCam

In [ ]:
# Set default to float32 to avoid precision errors
torch.set_default_dtype(torch.float32)
torch.backends.cudnn.benchmark = True
torch.backends.cudnn.deterministic = False

In [ ]:
# Check GPU availability
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")
    device = torch.device("cuda")
else:
    device = torch.device("cpu")
    print("Using CPU")
    # Enable full GPU memory utilization
if torch.cuda.is_available():
    # Clear GPU cache
    torch.cuda.empty_cache()
    # Set to use maximum GPU memory
    torch.cuda.set_per_process_memory_fraction(0.95)

CUDA available: True
CUDA device: NVIDIA A100-SXM4-40GB


In [ ]:
dataset = load_dataset("ThankGod/melanoma")

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
image_processor = AutoImageProcessor.from_pretrained("WinKawaks/vit-tiny-patch16-224")

Fast image processor class <class 'transformers.models.vit.image_processing_vit_fast.ViTImageProcessorFast'> is available for this model. Using slow image processor class. To use the fast image processor class set `use_fast=True`.


In [ ]:
# Preprocessing function
def transform(examples):
    images = [example for example in examples["image"]]
    processed_images = image_processor(
        images,
        return_tensors="pt",
        padding=True
    )["pixel_values"]

    return {"pixel_values": processed_images}

In [ ]:
# Apply transformation
print("Preprocessing dataset...")
dataset = dataset.map(
    transform,
    batched=True,
    batch_size=32,
    num_proc=2
)

Preprocessing dataset...


In [ ]:
# Format the dataset to PyTorch tensors
dataset.set_format(type="torch", columns=["pixel_values", "label"])

In [ ]:
# Class names for the dataset (adjust if needed)
class_names = ["Actinic keratosis", "Benign keratosis", "Dermatofibroma", "Melanocytic nevus", "Vascular lesion"]
# 0 -> Actinic keratosis
# 1 -> Benign keratosis
# 2 -> Dermatofibroma
# 3 -> Melanocytic nevus
# 4 -> Vascular lesion

In [ ]:
model = ViTForImageClassification.from_pretrained(
    "WinKawaks/vit-tiny-patch16-224",
    num_labels=5,
    id2label={i: class_names[i] for i in range(5)},
    label2id={class_names[i]: i for i in range(5)},
    ignore_mismatched_sizes=True,
    torch_dtype=torch.float32,
    output_attentions=True,  # Important for visualization
    attn_implementation="eager"  # Fix the warning
)


Some weights of ViTForImageClassification were not initialized from the model checkpoint at WinKawaks/vit-tiny-patch16-224 and are newly initialized because the shapes did not match:
- classifier.bias: found shape torch.Size([1000]) in the checkpoint and torch.Size([5]) in the model instantiated
- classifier.weight: found shape torch.Size([1000, 192]) in the checkpoint and torch.Size([5, 192]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
# Make sure all model parameters are float32
for param in model.parameters():
    param.data = param.data.float()

In [ ]:
# Move model to device
model = model.to(device)
print(f"Model moved to {device}")

Model moved to cuda


In [ ]:
# Training arguments
training_args = TrainingArguments(
    output_dir="./vit-melanoma",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    gradient_accumulation_steps=1,
    eval_strategy="epoch",
    save_strategy="epoch",
    num_train_epochs=5,
    save_total_limit=1,
    learning_rate=5e-5,
    remove_unused_columns=False,
    report_to="tensorboard",
    logging_dir="./logs",
    logging_steps=10,
    fp16=False,
    bf16=False,
    dataloader_pin_memory=True,
    dataloader_num_workers=2,
    gradient_checkpointing=False,  # Disable for attention visualization
    optim="adamw_torch",
    warmup_steps=100,
    weight_decay=0.01,
    metric_for_best_model="f1",  # Use F1 score to select best model
    load_best_model_at_end=True,
)

In [ ]:
# Custom compute metrics function to get precision, recall, and F1
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    if isinstance(logits, tuple):  # Ensure logits are extracted properly
        logits = logits[0]

    predictions = np.argmax(logits, axis=-1)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, predictions, average='weighted'
    )
    accuracy = (predictions == labels).mean()
    return {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1
    }


In [ ]:
# Create Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    compute_metrics=compute_metrics,
)

In [ ]:
# clear cache for training
torch.cuda.empty_cache()

# Final GPU check before training
if torch.cuda.is_available():
    print(f"Starting training on GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU memory before training: {torch.cuda.memory_allocated(0) / 1024**2:.2f} MB / {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")
else:
    print("WARNING: GPU not available, training on CPU will be much slower!")



Starting training on GPU: NVIDIA A100-SXM4-40GB
GPU memory before training: 21.11 MB / 39.56 GB


In [ ]:
# Start training
print("Starting training...")
trainer.train()

Starting training...


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.316300,0.340248,0.876369,0.871844,0.876369,0.870758
2,0.154800,0.320230,0.889671,0.884656,0.889671,0.883646
3,0.051100,0.451252,0.899844,0.898664,0.899844,0.896061
4,0.004500,0.460853,0.906886,0.905473,0.906886,0.905790
5,0.001800,0.494241,0.916275,0.916049,0.916275,0.915720


TrainOutput(global_step=4475, training_loss=0.15910980422182436, metrics={'train_runtime': 483.8216, 'train_samples_per_second': 147.968, 'train_steps_per_second': 9.249, 'total_flos': 3.572589637087027e+17, 'train_loss': 0.15910980422182436, 'epoch': 5.0})

In [ ]:
# Save the trained model
model_save_path = "./vit_melanoma_trained"
trainer.save_model(model_save_path)

# Save the image processor as well
image_processor.save_pretrained(model_save_path)

print(f"Model saved to {model_save_path}")

Model saved to ./vit_melanoma_trained


In [ ]:
# Evaluate the model
print("Evaluating model...")
eval_results = trainer.evaluate()
print(f"Evaluation results: {eval_results}")

Evaluating model...


Evaluation results: {'eval_loss': 0.49424123764038086, 'eval_accuracy': 0.9162754303599374, 'eval_precision': 0.9160492299141477, 'eval_recall': 0.9162754303599374, 'eval_f1': 0.9157199340214355, 'eval_runtime': 12.7671, 'eval_samples_per_second': 100.101, 'eval_steps_per_second': 3.133, 'epoch': 5.0}


In [ ]:
# Get predictions on test set for detailed analysis
test_dataloader = DataLoader(dataset["test"], batch_size=16)
test_predictions = []
test_labels = []
attention_weights = []

In [ ]:
model.eval()
with torch.no_grad():
    for batch in test_dataloader:
        inputs = batch["pixel_values"].to(device)
        labels = batch["label"].to(device)

        outputs = model(inputs, output_attentions=True)
        predictions = torch.argmax(outputs.logits, dim=-1)

        # Accumulate predictions and labels instead of extending
        test_predictions.append(predictions.cpu().numpy())
        test_labels.append(labels.cpu().numpy())

        # Store attention weights from the last layer (most informative)
        batch_attentions = outputs.attentions[-1].cpu().numpy()
        attention_weights.append(batch_attentions)


In [ ]:
# Concatenate predictions and labels after the loop
test_predictions = np.concatenate(test_predictions)
test_labels = np.concatenate(test_labels)

In [ ]:
# Generate confusion matrix
cm = confusion_matrix(test_labels, test_predictions)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
plt.title('Confusion Matrix')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.savefig('confusion_matrix.png')
plt.close()

In [ ]:
# Alpha precision and recall calculation (per-class metrics)
precision_per_class = []
recall_per_class = []

for class_idx in range(len(class_names)):
    class_preds = (test_predictions == class_idx)
    class_truth = (test_labels == class_idx)

    true_positives = np.logical_and(class_preds, class_truth).sum()
    false_positives = np.logical_and(class_preds, np.logical_not(class_truth)).sum()
    false_negatives = np.logical_and(np.logical_not(class_preds), class_truth).sum()

    precision = true_positives / (true_positives + false_positives) if (true_positives + false_positives) > 0 else 0
    recall = true_positives / (true_positives + false_negatives) if (true_positives + false_negatives) > 0 else 0

    precision_per_class.append(precision)
    recall_per_class.append(recall)

In [ ]:
# Print per-class metrics
print("\nPer-class Metrics:")
for i, class_name in enumerate(class_names):
    print(f"{class_name}: Precision = {precision_per_class[i]:.4f}, Recall = {recall_per_class[i]:.4f}")



Per-class Metrics:
Actinic keratosis: Precision = 0.8116, Recall = 0.7467
Benign keratosis: Precision = 0.7561, Recall = 0.7635
Dermatofibroma: Precision = 1.0000, Recall = 0.6364
Melanocytic nevus: Precision = 0.9558, Recall = 0.9637
Vascular lesion: Precision = 0.9583, Recall = 0.9583


In [ ]:
# Attention Visualization Functions
def get_attention_maps(input_image, model, image_processor):
    """Extract attention maps from the model for a single image"""
    # Preprocess the image
    image_tensor = image_processor(input_image, return_tensors="pt")["pixel_values"].to(device)

    # Forward pass
    with torch.no_grad():
        outputs = model(image_tensor, output_attentions=True)

    # Get attention maps from all layers
    attention_maps = [attn.squeeze(0).cpu().numpy() for attn in outputs.attentions]

    # Get the prediction
    prediction = torch.argmax(outputs.logits, dim=-1).item()
    confidence = torch.softmax(outputs.logits, dim=-1)[0][prediction].item()

    return attention_maps, prediction, confidence

def plot_attention_maps(image, attention_maps, prediction, confidence, class_names, save_path=None):
    """Plot attention maps for a given image"""
    # Get the attention map from the last layer (most informative for classification)
    last_layer_attention = attention_maps[-1]

    # Average attention across all heads
    averaged_attention = np.mean(last_layer_attention, axis=0)

    # Determine patch size from model configuration
    num_patches = int(np.sqrt(averaged_attention.shape[-1] - 1))  # Subtract 1 for CLS token

    # Reshape attention to match image patches (excluding CLS token)
    attention_for_patches = averaged_attention[0, 1:].reshape(num_patches, num_patches)

    # Create the figure
    fig, ax = plt.subplots(1, 2, figsize=(15, 7))

    # Plot original image
    ax[0].imshow(image)
    ax[0].set_title(f"Original Image\nPredicted: {class_names[prediction]} ({confidence:.2f})")
    ax[0].axis('off')

    # Plot attention map
    im = ax[1].imshow(attention_for_patches, cmap='hot')
    ax[1].set_title("Attention Map (Averaged Across Heads)")
    ax[1].axis('off')

    # Add colorbar
    plt.colorbar(im, ax=ax[1])

    if save_path:
        plt.savefig(save_path)
    plt.close()

def visualize_sample_attentions(dataset, model, image_processor, class_names, num_samples=5):
    """Visualize attention maps for a few sample images"""
    # Get some sample images
    test_dataset = dataset["test"]
    indices = [int(i) for i in np.random.choice(len(test_dataset), num_samples, replace=False)]

    for i, idx in enumerate(indices):
        # Get the pixel_values
        pixel_values = test_dataset[int(idx)]["pixel_values"]
        label = test_dataset[int(idx)]["label"]

        # Inverse transform pixel_values to PIL Image
        inverse_transform = transforms.ToPILImage()
        original_image = inverse_transform(pixel_values.squeeze(0))

        # Get attention maps
        attention_maps, prediction, confidence = get_attention_maps(original_image, model, image_processor)

        # Plot and save
        plot_attention_maps(
            original_image,
            attention_maps,
            prediction,
            confidence,
            class_names,
            save_path=f"attention_sample_{i}.png"
        )

        print(f"Sample {i}: Predicted {class_names[prediction]} with confidence {confidence:.4f}, True label: {class_names[label]}")


In [ ]:
# Visualize sample attentions
print("\nGenerating attention visualizations...")
visualize_sample_attentions(dataset, model, image_processor, class_names)



Generating attention visualizations...
Sample 0: Predicted Actinic keratosis with confidence 0.9766, True label: Benign keratosis
Sample 1: Predicted Melanocytic nevus with confidence 0.4727, True label: Melanocytic nevus
Sample 2: Predicted Actinic keratosis with confidence 0.8720, True label: Melanocytic nevus
Sample 3: Predicted Melanocytic nevus with confidence 0.9476, True label: Vascular lesion
Sample 4: Predicted Vascular lesion with confidence 0.3870, True label: Melanocytic nevus


In [ ]:
# Create a function to compute Grad-CAM for explainability
def compute_gradcam(model, input_image, target_category=None):
    """Compute Grad-CAM visualization for explainability"""
    from captum.attr import LayerGradCam

    # Preprocess the image
    input_tensor = image_processor(input_image, return_tensors="pt")["pixel_values"].to(device)
    input_tensor.requires_grad = True

    # Define the layer to use for Grad-CAM (typically the last convolutional layer)
    # For ViT, we use the output of the transformer blocks
    layer = model.vit.encoder.layer[-1]

    # Create GradCAM object
    grad_cam = LayerGradCam(model, layer)

    # If no target category specified, use the predicted class
    if target_category is None:
        with torch.no_grad():
            output = model(input_tensor)
            target_category = output.logits.argmax(dim=1).item()

    # Compute Grad-CAM
    attribution = grad_cam.attribute(input_tensor, target=target_category)

    # Reshape attribution to match image size
    attribution = attribution.sum(dim=1).cpu().detach().numpy()

    return attribution, target_category

# Function to visualize Grad-CAM
def visualize_gradcam(image, attribution, prediction, class_names, save_path=None):
    """Visualize Grad-CAM attribution"""
    # Create figure
    fig, ax = plt.subplots(1, 2, figsize=(15, 7))

    # Plot original image
    ax[0].imshow(image)
    ax[0].set_title(f"Original Image\nPredicted: {class_names[prediction]}")
    ax[0].axis('off')

    # Plot Grad-CAM
    cam = attribution[0]
    # Normalize between 0 and 1
    cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
    ax[1].imshow(image)
    cam_resized = transforms.Resize((image.height, image.width))(torch.tensor(cam)[None])[0]
    ax[1].imshow(cam_resized, cmap='jet', alpha=0.6)
    ax[1].set_title(f"Grad-CAM for {class_names[prediction]}")
    ax[1].axis('off')

    if save_path:
        plt.savefig(save_path)
    plt.close()

# Visualize Grad-CAM for a few samples
def visualize_sample_gradcams(dataset, model, image_processor, class_names, num_samples=5):
    """Visualize Grad-CAM for sample images"""
    # Only try this if captum is available
    try:
        from captum.attr import LayerGradCam
        # Get some sample images
        test_dataset = dataset["test"]
        indices = np.random.choice(len(test_dataset), num_samples, replace=False)

        for i, idx in enumerate(indices):
            # Get the original image
            original_image = test_dataset[idx]["image"]

            # Get GradCAM
            attribution, prediction = compute_gradcam(model, original_image)

            # Plot and save
            visualize_gradcam(
                original_image,
                attribution,
                prediction,
                class_names,
                save_path=f"gradcam_sample_{i}.png"
            )

            print(f"GradCAM Sample {i}: Predicted {class_names[prediction]}")
    except ImportError:
        print("Captum not installed. Skipping Grad-CAM visualization.")

In [ ]:
# Try to visualize Grad-CAM
try:
    print("\nGenerating Grad-CAM visualizations...")
    visualize_sample_gradcams(dataset, model, image_processor, class_names)
except Exception as e:
    print(f"Error generating Grad-CAM: {e}")

print("\nEvaluation and visualization complete!")


Generating Grad-CAM visualizations...
Error generating Grad-CAM: Wrong key type: '735' of type '<class 'numpy.int64'>'. Expected one of int, slice, range, str or Iterable.

Evaluation and visualization complete!
